# Data Analysis con Pandas

> Pandas es una librería de Python escrita como extensión de NumPy para manipulación y análisis de datos.

Sitio oficial: [pandas.python.org](https://pandas.pydata.org/)
Documentación Oficial: [pandas.pydata.org/pandas-docs/stable/](https://pandas.pydata.org/pandas-docs/stable/)

## Fusión de DataFrames

In [1]:
import pandas as pd

En este notebook veremos cómo combinar varios DataFrame, ya sea fusionándolos horizontalmente o concatenándolos verticalmente.

### Introducción a la fusión de DataFrames

Antes de entrar en el código, repasaremos algunos conceptos básicos de la teoría relacional y las convenciones del lenguaje. Incluiré una imagen para ilustrar algunos conceptos.

![Venn Diagram](../images/venn-diagram.png)


Un diagrama de Venn se usa tradicionalmente para mostrar la pertenencia a un conjunto. Por ejemplo, el círculo de la izquierda representa a la población estudiantil de una universidad. El círculo de la derecha representa a la población del personal de la universidad. Y la región superpuesta en el centro incluye a todos los estudiantes que también son personal. Quizás estos estudiantes imparten tutorías para un curso, califican tareas o participan en la realización de experimentos de investigación.

Así pues, este diagrama muestra dos poblaciones de las que podríamos tener datos, pero existe una superposición entre ellas.populations.

En pandas podemos considerar el caso en el que tengamos estas dos poblaciones como índices en DataFrames separados, quizás con la etiqueta "Nombre de la persona". Entonces, podemos unir los DataFrames y obtener toda la información disponible sobre ellas.

- En terminología de bases de datos, esto se denomina unión externa completa (full outer join).
- En teoría de conjuntos, se denomina unión.
- En un diagrama de Venn, representa a todos los elementos de cualquier círculo.

Aquí hay una imagen de cómo se vería esto en un diagrama de Venn.

![Union](../images/union.png)

También nos puede interesar conocer aquellas personas que son personal, pero que también son estudiantes.

- En terminología de bases de datos, esto se denomina unión interna (inner join).

- En teoría de conjuntos, intersección.

- En el diagrama de Venn, se representa como las partes superpuestas de cada círculo.

Así es como se ve:
![Intersection](../images/intersection.png)


### Fusión utilizando la función `merge()`

Con ese contexto, veamos un ejemplo de cómo haríamos esto en pandas, donde usaríamos la función merge.

In [2]:
staff_df = pd.DataFrame([{'Name': 'Kelly', 'Role': 'Director of HR'},
                         {'Name': 'Sally', 'Role': 'Course liasion'},
                         {'Name': 'James', 'Role': 'Grader'}])
staff_df = staff_df.set_index('Name')

staff_df.head()

,Role
Name,
Kelly,Director of HR
Sally,Course liasion
James,Grader


In [3]:
student_df = pd.DataFrame([{'Name': 'James', 'School': 'Business'},
                           {'Name': 'Mike', 'School': 'Law'},
                           {'Name': 'Sally', 'School': 'Engineering'}])
student_df = student_df.set_index('Name')

student_df.head()

,School
Name,
James,Business
Mike,Law
Sally,Engineering


> Nota:
>
> Existe cierta superposición entre estos DataFrames, ya que James y Sally son estudiantes, pero también son personal. Es importante destacar que ambos DataFrames están indexados por el valor que queremos usar para combinarlos: `Name`.


#### merge how=outer

Para unirlos, llamaríamos a la función `merge()`, pasando el DataFrame de la izquierda y el de la derecha, e indicándole que queremos usar una unión externa. Queremos usar los índices de las columnas izquierda y derecha como columnas de unión.

In [4]:
pd.merge(staff_df,
         student_df,
         how='outer',
         left_index=True,
         right_index=True)

,Role,School
Name,,
James,Grader,Business
Kelly,Director of HR,NaN
Mike,NaN,Law
Sally,Course liasion,Engineering


> Observamos que en el DataFrame resultante todas las personas están listados.
> Dado que Mike no tiene un rol asignado y John no pertenece a ninguna escuela, esas celdas se muestran como valores faltantes `NaN`.

#### merge how=inner


Si quisiéramos obtener la intersección, es decir, solo aquellos que son estudiantes y también son personal, podríamos establecer el atributo `how` en `inner`. Nuevamente, configuramos los índices `left` y `right` como verdaderos, ya que son las columnas de unión.

In [5]:
pd.merge(staff_df,
         student_df,
         how='inner',
         left_index=True,
         right_index=True)

,Role,School
Name,,
Sally,Course liasion,Engineering
James,Grader,Business


> Observamos que el DataFrame resultante solo contiene a James y Sally.

#### merge how=left

Ahora bien, existen otros dos casos de uso comunes al combinar DataFrames, y ambos son ejemplos de lo que llamaríamos adición de conjuntos.

- El primero es cuando queremos obtener una lista de todo el personal, independientemente de si son o no estudiantes. Pero si son estudiantes, también querríamos obtener sus datos académicos. Para ello, usaríamos un LEFT JOIN. Es importante tener en cuenta el orden de los DataFrames en esta función: el primer DataFrame es el de la izquierda y el segundo, el de la derecha.

In [6]:
pd.merge(staff_df,
         student_df,
         how='left',
         left_index=True,
         right_index=True)

,Role,School
Name,,
Kelly,Director of HR,NaN
Sally,Course liasion,Engineering
James,Grader,Business


#### merge how=right

Ahora, queremos una lista de todos los estudiantes y sus funciones si también fueran personal docente. Para ello, usaríamos una unión derecha (right join).

In [7]:
pd.merge(staff_df,
         student_df,
         how='right',
         left_index=True,
         right_index=True)

,Role,School
Name,,
James,Grader,Business
Mike,NaN,Law
Sally,Course liasion,Engineering


#### merge on=

También podemos hacerlo de otra manera.

El método `merge` tiene otros parámetros interesantes. Primero, no es necesario usar índices para la unión, también se pueden usar columnas. Tenemos un parámetro llamado "on", y podemos asignar como columna de unión una columna que ambos dataframes tengan.

In [8]:
# Eliminamos los índices de los dos DataFrames
staff_df = staff_df.reset_index()
student_df = student_df.reset_index()

pd.merge(staff_df,
         student_df,
         how='right',
         on='Name')

,Name,Role,School
0,James,Grader,Business
1,Mike,NaN,Law
2,Sally,Course liasion,Engineering


#### Conflicto en nombre de columnas al mergear DataFrames

In [9]:
staff_df = pd.DataFrame([{'Name': 'Kelly',
                          'Role': 'Director of HR',
                          'Location': 'State Street'},
                         {'Name': 'Sally',
                          'Role': 'Course liasion',
                          'Location': 'Washington Avenue'},
                         {'Name': 'James',
                          'Role': 'Grader',
                          'Location': 'Washington Avenue'}])
student_df = pd.DataFrame([{'Name': 'James',
                            'School': 'Business',
                            'Location': '1024 Billiard Avenue'},
                           {'Name': 'Mike',
                            'School': 'Law',
                            'Location': 'Fraternity House #22'},
                           {'Name': 'Sally',
                            'School': 'Engineering',
                            'Location': '512 Wilson Crescent'}])

En el DataFrame del personal, esta es la ubicación de la oficina donde podemos encontrar a cada empleado. Podemos ver que el Director de Recursos Humanos está en State Street, mientras que los dos estudiantes están en Washington Avenue. Sin embargo, en el DataFrame de los estudiantes, la información de ubicación corresponde a su domicilio particular.

La función `merge` conserva esta información, pero agrega un `_x` o `_y` para diferenciar qué índice corresponde a cada columna de datos. El sufijo `_x` siempre corresponde a la información del DataFrame de la izquierda, y el sufijo `_y` siempre a la del DataFrame de la derecha.

In [10]:
pd.merge(staff_df,
         student_df,
         how='left',
         on='Name')

,Name,Role,Location_x,School,Location_y
0,Kelly,Director of HR,State Street,NaN,NaN
1,Sally,Course liasion,Washington Avenue,Engineering,512 Wilson Crescent
2,James,Grader,Washington Avenue,Business,1024 Billiard Avenue


> Nota:
>
> En la salida, podemos ver las columnas Location_x y Location_y. Location_x se refiere a la columna Location del dataframe de la izquierda, que contiene el dato del personal, y Location_y se refiere a la columna Location del dataframe de la derecha, que contiene el dato de los estudiantes.

#### Conflictos en columnas-índices al mergear DataFrames

Hablemos de conflictos vinculados con multi-indexación y columnas múltiples.

Es muy probable que los nombres de pila de estudiantes y personal coincidan, pero no necesariamente sus apellidos. En este caso, utilizamos para unir ambos DataFrames mediante el parámetro `on` una lista de las columnas necesarias. Recordemos que el nombre o los nombres de columna asignados al parámetro `on` deben existir en ambos DataFrames.

In [11]:
staff_df = pd.DataFrame([{'First Name': 'Kelly',
                          'Last Name': 'Desjardins',
                          'Role': 'Director of HR'},
                         {'First Name': 'Sally',
                          'Last Name': 'Brooks',
                          'Role': 'Course liasion'},
                         {'First Name': 'James',
                          'Last Name': 'Wilde',
                          'Role': 'Grader'}])
student_df = pd.DataFrame([{'First Name': 'James',
                            'Last Name': 'Hammond',
                            'School': 'Business'},
                           {'First Name': 'Mike',
                            'Last Name': 'Smith',
                            'School': 'Law'},
                           {'First Name': 'Sally',
                            'Last Name': 'Brooks',
                            'School': 'Engineering'}])

pd.merge(staff_df,
         student_df,
         how='inner',
         on=['First Name', 'Last Name'])

,First Name,Last Name,Role,School
0,Sally,Brooks,Course liasion,Engineering


> Nota:
>
> Como podemos observar, James Wilde y James Hammond no coinciden en ambas claves, ya que tienen apellidos diferentes. Por lo tanto, esperamos que una unión interna no incluya a estas personas en el resultado, y que solo se conserve Sally Brooks.

### Fusión utilizando la función `concat()`

Unir dataframes mediante `merge` es muy común, y por ello necesitaremos saber cómo extraer datos de diferentes fuentes, limpiarlos y unirlos para su análisis. Esto es fundamental no solo en pandas, sino también en las tecnologías de bases de datos.

Si consideramos la fusión como una unión "horizontal", es decir, unimos los datos basándonos en valores similares en una columna de dos dataframes, entonces la concatenación es una unión "vertical".

Veamos un ejemplo. Supongamos que tenemos un conjunto de datos que registra información a lo largo de los años. El registro de cada año es un archivo CSV independiente, y cada archivo CSV contiene exactamente las mismas columnas.

¿Qué sucede si queremos reunir todos los datos de todos los años? Podemos concatenarlos.

Veamos los datos del College Scorecard del Departamento de Educación de EE. UU. Contienen información de cada universidad estadounidense sobre la finalización de estudios, la deuda estudiantil, los ingresos posteriores a la graduación, etc. Los datos se almacenan en archivos CSV separados, cada uno con el registro de un año. Digamos que queremos los registros de 2011 a 2013; primero creamos tres dataframes, cada uno con el registro de un año. Y, dado que los archivos CSV con los que estamos trabajando están desordenados, quiero suprimir algunos de los mensajes de advertencia de Jupyter y simplemente indicarle a `read_csv` que ignore las líneas incorrectas, así que vamos a iniciar la celda con una función mágica llamada `%%capture`

In [12]:
%%capture
df_2011 = pd.read_csv("../data/college_scorecard/MERGED2010_11_PP.csv", on_bad_lines='skip')
df_2012 = pd.read_csv("../data/college_scorecard/MERGED2011_12_PP.csv", on_bad_lines='skip')
df_2013 = pd.read_csv("../data/college_scorecard/MERGED2012_13_PP.csv", on_bad_lines='skip')

In [13]:
df_2011.head(3)

,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,OMAWDP8_NOTFIRSTTIME_POOLED_SUPP,OMENRUP_NOTFIRSTTIME_POOLED_SUPP,OMENRYP_FULLTIME_POOLED_SUPP,OMENRAP_FULLTIME_POOLED_SUPP,OMAWDP8_FULLTIME_POOLED_SUPP,OMENRUP_FULLTIME_POOLED_SUPP,OMENRYP_PARTTIME_POOLED_SUPP,OMENRAP_PARTTIME_POOLED_SUPP,OMAWDP8_PARTTIME_POOLED_SUPP,OMENRUP_PARTTIME_POOLED_SUPP
0,100654,100200,1002,Alabama A & M University,Normal,AL,35762,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100663,105200,1052,University of Alabama at Birmingham,Birmingham,AL,35294-0110,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100690,2503400,25034,Amridge University,Montgomery,AL,36117-3553,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
len(df_2011), len(df_2012), len(df_2013)

(7414, 15235, 7793)

> Nota:
>
> Resulta un tanto sorprendente que el número de escuelas en la evaluación de 2012 sea casi el doble que en los dos años siguientes. Pero no nos preocupemos por eso. En su lugar, coloquemos los tres dataframes en una lista llamada `frames` y pasémosla a la función `concat()`. Veamos qué resultado se obtiene.



In [15]:
frames = [df_2011, df_2012, df_2013]
pd.concat(frames)

,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,OMAWDP8_NOTFIRSTTIME_POOLED_SUPP,OMENRUP_NOTFIRSTTIME_POOLED_SUPP,OMENRYP_FULLTIME_POOLED_SUPP,OMENRAP_FULLTIME_POOLED_SUPP,OMAWDP8_FULLTIME_POOLED_SUPP,OMENRUP_FULLTIME_POOLED_SUPP,OMENRYP_PARTTIME_POOLED_SUPP,OMENRAP_PARTTIME_POOLED_SUPP,OMAWDP8_PARTTIME_POOLED_SUPP,OMENRUP_PARTTIME_POOLED_SUPP
0,100654.0,100200,1002,Alabama A & M University,Normal,AL,35762,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100663.0,105200,1052,University of Alabama at Birmingham,Birmingham,AL,35294-0110,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100690.0,2503400,25034,Amridge University,Montgomery,AL,36117-3553,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100706.0,105500,1055,University of Alabama in Huntsville,Huntsville,AL,35899,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100724.0,100500,1005,Alabama State University,Montgomery,AL,36104-0271,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7788,47691101.0,4205801,42058,SAE Institute of Technology San Francisco,Emeryville,CA,94608,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7789,47701101.0,10145905,1459,Strayer University-Bloomington Campus,Bloomington,MN,554311411,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7790,47702001.0,10145903,1459,Strayer University-Schaumburg Campus,Schaumburg,IL,601735081,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7791,47702002.0,10145902,1459,Strayer University-Downers Grove Campus,Downers Grove,IL,605151169,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


> Nota
>
> Como puede verse, tenemos más observaciones en un dataframe y las columnas permanecen iguales. Si nos desplazamos hasta el final de la salida, vemos que hay un total de 30442 filas después de concatenar los tres dataframes.

Sumemos el número de filas de los tres dataframes y veamos si ambos resultados coinciden.

In [16]:
len(df_2011) + len(df_2012) + len(df_2013)

30442

Los dos números coinciden. Esto significa que la concatenación fue exitosa. Pero un momento, ahora que todos los datos están concatenados, ¡ya no sabemos a qué año pertenecen las observaciones! La función `concat` tiene un parámetro que resuelve este problema: el parámetro `keys`. Podemos establecer un nivel adicional de índices; para ello, pasamos una lista de claves que corresponden a los dataframes.

In [17]:
pd.concat(frames, keys=['2011','2012','2013'])

UNITID     OPEID OPEID6  \
2011 0       100654.0    100200   1002   
     1       100663.0    105200   1052   
     2       100690.0   2503400  25034   
     3       100706.0    105500   1055   
     4       100724.0    100500   1005   
...               ...       ...    ...   
2013 7788  47691101.0   4205801  42058   
     7789  47701101.0  10145905   1459   
     7790  47702001.0  10145903   1459   
     7791  47702002.0  10145902   1459   
     7792  47702003.0  10145906   1459   

                                              INSTNM           CITY STABBR  \
2011 0                      Alabama A & M University         Normal     AL   
     1           University of Alabama at Birmingham     Birmingham     AL   
     2                            Amridge University     Montgomery     AL   
     3           University of Alabama in Huntsville     Huntsville     AL   
     4                      Alabama State University     Montgomery     AL   
...                                              ...            ...    ...   
2013 7788  SAE Institute of Technology San Francisco     Emeryville     CA   
     7789      Strayer University-Bloomington Campus    Bloomington     MN   
     7790       Strayer University-Schaumburg Campus     Schaumburg     IL   
     7791    Strayer University-Downers Grove Campus  Downers Grove     IL   
     7792           Strayer University-Aurora Campus         Aurora     IL   

                  ZIP  ACCREDAGENCY INSTURL NPCURL  ...  \
2011 0          35762           NaN     NaN    NaN  ...   
     1     35294-0110           NaN     NaN    NaN  ...   
     2     36117-3553           NaN     NaN    NaN  ...   
     3          35899           NaN     NaN    NaN  ...   
     4     36104-0271           NaN     NaN    NaN  ...   
...               ...           ...     ...    ...  ...   
2013 7788       94608           NaN     NaN    NaN  ...   
     7789   554311411           NaN     NaN    NaN  ...   
     7790   601735081           NaN     NaN    NaN  ...   
     7791   605151169           NaN     NaN    NaN  ...   
     7792   605066220           NaN     NaN    NaN  ...   

          OMAWDP8_NOTFIRSTTIME_POOLED_SUPP OMENRUP_NOTFIRSTTIME_POOLED_SUPP  \
2011 0                                 NaN                              NaN   
     1                                 NaN                              NaN   
     2                                 NaN                              NaN   
     3                                 NaN                              NaN   
     4                                 NaN                              NaN   
...                                    ...                              ...   
2013 7788                              NaN                              NaN   
     7789                              NaN                              NaN   
     7790                              NaN                              NaN   
     7791                              NaN                              NaN   
     7792                              NaN                              NaN   

          OMENRYP_FULLTIME_POOLED_SUPP OMENRAP_FULLTIME_POOLED_SUPP  \
2011 0                             NaN                          NaN   
     1                             NaN                          NaN   
     2                             NaN                          NaN   
     3                             NaN                          NaN   
     4                             NaN                          NaN   
...                                ...                          ...   
2013 7788                          NaN                          NaN   
     7789                          NaN                          NaN   
     7790                          NaN                          NaN   
     7791                          NaN                          NaN   
     7792                          NaN                          NaN   

          OMAWDP8_FULLTIME_POOLED_SUPP OMENRUP_FULLTIME_POOLED

Ahora que tenemos los índices del año, sabemos a qué año pertenecen las observaciones.

> Es importante saber que la concatenación tiene dos métodos: interno y externo. Si concatenas dos dataframes con columnas diferentes y eliges el método externo, algunas celdas contendrán valores NaN. Si eliges el método interno, se eliminarán algunas observaciones debido a los valores NaN. Esto es similar a las combinaciones izquierda y derecha de la función `merge()`.

## Conclusión

Ahora ya sabes cómo 'mergear' y 'concatenar' conjuntos de datos. Estas funciones te resultarán muy útiles para combinar datos y obtener resultados más complejos, así como para realizar análisis. Un buen conocimiento de cómo fusionar datos es fundamental para la adquisición, limpieza y manipulación de datos. Es importante saber cómo unir diferentes conjuntos de datos rápidamente y conocer las distintas opciones disponibles para esta unión. Te recomiendo consultar la documentación de pandas sobre la unión y concatenación de datos.